# 00 Data Processing


In [3]:
%load_ext autoreload
%autoreload 2

import json
import pickle
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
from numpy import linalg as la

from config import (
    DATA_DIR,
    MOLECULE_DIR,
    PARAMETERS_DIR,
    RAW_MATRICES_DIR,
    PROCESSED_DATAFRAMES_DIR
)


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [4]:
SV_HAMILTONIAN_DIR = RAW_MATRICES_DIR / "SV_hamiltonian"
SV_SPIN_DIR = RAW_MATRICES_DIR / "SV_spin"
PYSCF_FCI_PATH = RAW_MATRICES_DIR / "pyscf_fci.jsonl"


def read_jsonl(path):
    with path.open() as f:
        for line in f:
            yield json.loads(line)


def complex_matrix(real, imag):
    return np.array(real) + 1j * np.array(imag)


def load_sv_hamiltonian_records(path):
    records = []

    for file in sorted(path.glob("*.jsonl")):
        for row in read_jsonl(file):
            H = complex_matrix(row["h_matrix_real"], row["h_matrix_imag"])
            S = complex_matrix(row["s_matrix_real"], row["s_matrix_imag"])

            records.append({
                "molecule": row["molecule"],
                "active_space": row["active_space"],
                "ansatz": row["ansatz"],
                "expansion": row["expansion"],
                "H": H,
                "S": S,
                "qse_dim": H.shape[0],
            })

    return pd.DataFrame(records)


def load_spin_records(path):
    records = []

    for file in sorted(path.glob("*.jsonl")):
        for row in read_jsonl(file):
            S2 = complex_matrix(row["h_matrix_real"], row["h_matrix_imag"])
            S_spin = complex_matrix(row["s_matrix_real"], row["s_matrix_imag"])

            records.append({
                "molecule": row["molecule"],
                "active_space": row["active_space"],
                "ansatz": row["ansatz"],
                "expansion": row["expansion"],
                "S2": S2,
                "S_spin": S_spin,
            })

    return pd.DataFrame(records)


def load_pyscf_fci_records(path):
    records = []

    for row in read_jsonl(path):
        key, matrix = next(iter(row.items()))
        molecule, active_space, expansion = key.rsplit("_", 2)
        matrix = np.array(matrix)

        records.append({
            "molecule": molecule,
            "active_space": active_space,
            "expansion": expansion,
            "pyscf_fci_matrix": matrix,
            "pyscf_fci_shape": matrix.shape,
        })

    return pd.DataFrame(records)


df_sv_hamiltonian = load_sv_hamiltonian_records(SV_HAMILTONIAN_DIR)
df_sv_spin = load_spin_records(SV_SPIN_DIR)
df_pyscf_fci = load_pyscf_fci_records(PYSCF_FCI_PATH)

df_sv = df_sv_hamiltonian.merge(
    df_sv_spin,
    on=["molecule", "active_space", "ansatz", "expansion"],
    how="left",
)

df_sv = df_sv.merge(
    df_pyscf_fci,
    on=["molecule", "active_space", "expansion"],
    how="left",
)

df_sv = df_sv.sort_values(
    ["molecule", "active_space", "ansatz", "expansion"]
).reset_index(drop=True)

df_sv


,molecule,active_space,ansatz,expansion,H,S,qse_dim,S2,S_spin,pyscf_fci_matrix,pyscf_fci_shape
0,Acetamide,2e2o,1UpCCGSDSinglet,singlet,"[[(-802.5663414572622+0j), (-4.706692634862637...","[[(3.9097041888243123+0j), (0.0229245233142809...",4,"[[(1.27675647831893e-14+0j), (-4.3368086899420...","[[(3.9097041888243123+0j), (0.0229245233142809...","[[-0.15005503267467155, 0.07060303732223218, -...","(6, 3)"
1,Acetamide,2e2o,1UpCCGSDSinglet,triplet,"[[(-0.008069881124952466+0j), (0.1930299587744...","[[(3.9350963402490224e-05+0j), (-0.00094126725...",4,"[[(7.870192680212162e-05+0j), (-0.001882534507...","[[(3.9350963402490224e-05+0j), (-0.00094126725...","[[0.0, 2.472254987467345e-16, 0.0], [1.0, 0.0,...","(6, 3)"
2,Acetamide,2e2o,UCCGSD,singlet,"[[(-802.5658955566977+0j), (-4.714686911530124...","[[(3.909702017207626+0j), (0.02296346038353209...",4,"[[(1.2961853812498703e-14+0j), (-4.33680868994...","[[(3.909702017207626+0j), (0.02296346038353209...","[[-0.15005503267467155, 0.07060303732223218, -...","(6, 3)"
3,Acetamide,2e2o,UCCGSD,triplet,"[[(-0.00809733319789395+0j), (0.19335947310663...","[[(3.9484827272451284e-05+0j), (-0.00094287405...",4,"[[(7.896965454190497e-05+0j), (-0.001885748112...","[[(3.9484827272451284e-05+0j), (-0.00094287405...","[[0.0, 2.472254987467345e-16, 0.0], [1.0, 0.0,...","(6, 3)"
4,Acetamide,2e2o,UCCSD,singlet,"[[(-802.5658955566977+0j), (-4.714686911530124...","[[(3.909702017207626+0j), (0.02296346038353209...",4,"[[(1.2961853812498703e-14+0j), (-4.33680868994...","[[(3.909702017207626+0j), (0.02296346038353209...","[[-0.15005503267467155, 0.07060303732223218, -...","(6, 3)"
...,...,...,...,...,...,...,...,...,...,...,...
1563,Uracil,6e6o,UCCGSD,triplet,"[[(-3.056612740692799e-10+0j), 0j, 0j, 0j, 0j,...","[[(7.535361223887094e-13+0j), 0j, 0j, 0j, 0j, ...",36,NaN,NaN,NaN,NaN
1564,Uracil,6e6o,UCCSD,singlet,"[[(-1628.146797538434+0j), 0j, 0j, 0j, 0j, 0j,...","[[(3.9991745183913965+0j), 0j, 0j, 0j, 0j, 0j,...",36,NaN,NaN,NaN,NaN
1565,Uracil,6e6o,UCCSD,triplet,"[[(-1.0753353762993356e-10+0j), 0j, 0j, 0j, 0j...","[[(2.6509350270487175e-13+0j), 0j, 0j, 0j, 0j,...",36,NaN,NaN,NaN,NaN
1566,Uracil,6e6o,UCCSDSinglet,singlet,"[[(-1628.1503087714625+0j), 0j, 0j, 0j, 0j, 0j...","[[(3.9991831439323717+0j), 0j, 0j, 0j, 0j, 0j,...",36,NaN,NaN,NaN,NaN


In [5]:
key_cols = ["molecule", "active_space", "ansatz", "expansion"]


def is_matrix(value):
    return isinstance(value, np.ndarray) and value.ndim == 2


def is_square(value):
    return is_matrix(value) and value.shape[0] == value.shape[1]


def is_hermitian(value, atol=1e-10):
    return is_square(value) and np.allclose(value, value.conj().T, atol=atol)


def is_missing(value):
    return not isinstance(value, np.ndarray) and pd.isna(value)


def missing_count(column):
    return int(df_sv[column].map(is_missing).sum())


duplicate_rows = df_sv[df_sv.duplicated(key_cols, keep=False)]

shape_ok = df_sv.apply(
    lambda row: (
        is_square(row["H"])
        and is_square(row["S"])
        and row["H"].shape == row["S"].shape
        and row["H"].shape[0] == row["qse_dim"]
    ),
    axis=1,
)

spin_shape_ok = df_sv.apply(
    lambda row: is_missing(row["S2"]) or (is_square(row["S2"]) and row["S2"].shape == row["H"].shape),
    axis=1,
)

hermitian_summary = pd.DataFrame({
    "matrix": ["H", "S", "S2", "S_spin"],
    "non_hermitian_rows": [
        int((~df_sv["H"].map(is_hermitian)).sum()),
        int((~df_sv["S"].map(is_hermitian)).sum()),
        int(df_sv["S2"].map(lambda value: False if is_missing(value) else not is_hermitian(value)).sum()),
        int(df_sv["S_spin"].map(lambda value: False if is_missing(value) else not is_hermitian(value)).sum()),
    ],
})

fidelity_summary = pd.DataFrame({
    "check": [
        "rows",
        "duplicate calculation keys",
        "missing spin rows",
        "missing PySCF rows",
        "bad H/S shape rows",
        "bad spin shape rows",
    ],
    "value": [
        len(df_sv),
        len(duplicate_rows),
        missing_count("S2"),
        missing_count("pyscf_fci_matrix"),
        int((~shape_ok).sum()),
        int((~spin_shape_ok).sum()),
    ],
})

missing_by_active_space = (
    df_sv.assign(
        missing_spin=df_sv["S2"].map(is_missing),
        missing_pyscf=df_sv["pyscf_fci_matrix"].map(is_missing),
    )
    .groupby("active_space", as_index=False)[["missing_spin", "missing_pyscf"]]
    .sum()
)

print("Data fidelity summary")
print(fidelity_summary.to_string(index=False))

print("\nHermiticity checks")
print(hermitian_summary.to_string(index=False))

print("\nMissing joins by active space")
print(missing_by_active_space.to_string(index=False))

if not duplicate_rows.empty:
    print("\nDuplicate calculation keys")
    print(duplicate_rows[key_cols].sort_values(key_cols).to_string(index=False))
else:
    print("\nNo duplicate calculation keys found.")


Data fidelity summary
                     check  value
                      rows   1568
duplicate calculation keys      0
         missing spin rows    224
        missing PySCF rows    224
        bad H/S shape rows      0
       bad spin shape rows      0

Hermiticity checks
matrix  non_hermitian_rows
     H                   0
     S                   0
    S2                   0
S_spin                   0

Missing joins by active space
active_space  missing_spin  missing_pyscf
        2e2o             0              0
        2e3o             0              0
        4e3o             0              0
        4e4o             0              0
        4e5o             0              0
        6e5o             0              0
        6e6o           224            224

No duplicate calculation keys found.
